# Mixed radix FFT - Float

## Radix-2 SDF Butterfly

In [1]:
import numpy as np
from scipy.fft import fft, ifft

In [106]:
class radix2_PreAdder:
    """
    A class used to represent a hardware preadder of a radix-2 butterfly

    ...

    Attributes
    ----------
    input_a : float
        upper input
    input_b : float
        lower input
    output_add : float
        adder output
    output_sub : float
        subtractor output

    Methods
    -------
    calculate(self)
        Calculates the adder and the subtractor outputs 
    """
    def __init__(self):
        self.input_a = 0.0
        self.input_b = 0.0
        self.output_add = 0.0
        self.output_sub = 0.0


    def calculate(self):
        self.output_add = self.input_a + self.input_b
        self.output_sub = self.input_a - self.input_b
        

class radix2_Rotator:
    cnt = 0
    def __init__(self, stage_index, size):
        self.input = 0.0
        self.output = 0.0
        self.stage_index = stage_index
        # self.twiddleROM = (np.ones(2**(num_of_stages-stage_index))).astype(complex)
        self.twiddleROM = (np.ones(size)).astype(complex)
        self.half_len = size//2
        N = size
        for i in range(self.half_len):
            k = i * 2**(stage_index)
            self.twiddleROM[i+self.half_len] = np.exp(-1j*2*np.pi*k/N)
        
        # print(f"STAGE {stage_index}, twiddle = {self.twiddleROM}")

        # print(self.twiddleROM)

    def rotate(self, fifo_full_flag):
        if (fifo_full_flag): # dozvola da brojac vrti i cita redom twiddle faktore iz memorije
            # print(f"STAGE {self.stage_index}, FIFO FULL")
            self.output = self.input * self.twiddleROM[self.cnt]
            # print(f'STAGE {self.stage_index}, curr twiddle = {self.twiddleROM[self.cnt]}')
            if (self.cnt == self.half_len*2-1):
                self.cnt = 0
            else:
                self.cnt += 1
        

class Fifo:
    full = 0
    cnt = 0

    def __init__(self, depth):
        self.depth = depth
        self.buffer = (np.zeros(depth)).astype(complex)
        # print(self.depth)

    def is_full(self):
        return self.full
    
    def get_output(self):
        return self.buffer[-1]
    
    def shift(self, input_sample):
        self.cnt += 1
        self.buffer = np.roll(self.buffer, 1)
        # print("FIFO input samples = ", input_sample)
        self.buffer[0] = input_sample
        if (self.cnt > self.depth):
            self.full = 1
        else:
            self.full = 0



In [107]:
class radix2_SDF_stage:
    input_sample = 0.0
    output_sample = 0.0
    op_cnt = 0 # operation counter (counting how many add/subb operations are done)
    def __init__(self, stage_index, size):
        self.stage_index = stage_index
        self.size = size
        # self.num_of_stages = num_of_stages
        # self.num_of_samples = 2**(num_of_stages-stage_index)
        self.num_of_samples = size
        self.fifo = Fifo(size//2)
        self.pre_adder = radix2_PreAdder()
        self.rotator = radix2_Rotator(stage_index=stage_index, size=size)

    def isFifoFull(self):
        return self.fifo.is_full()
    
    def calculate(self):
        # print(f'STAGE {self.stage_index}, op_cnt = ', self.op_cnt)
        self.pre_adder.input_a = self.fifo.get_output()
        self.pre_adder.input_b = self.input_sample
        self.pre_adder.calculate()

        # print(f'STAGE {self.size}, add_out = {self.pre_adder.output_add}')
        # print(f'STAGE {self.size}, sub_out = {self.pre_adder.output_sub}')
        
        if (self.op_cnt//(self.num_of_samples/2)): ## other half of the input stream is comming
            self.output_sample = self.pre_adder.output_add
            self.fifo.shift(self.pre_adder.output_sub) 
        else:
            self.output_sample = self.fifo.get_output()
            self.fifo.shift(self.input_sample)

        self.rotator.input = self.output_sample
        self.rotator.rotate(self.isFifoFull())
        self.output_sample = self.rotator.output

        # print(f'stage_{self.stage_index} : {self.output_sample}')

        if (self.op_cnt == self.num_of_samples-1):
            self.op_cnt = 0
        else:
            self.op_cnt += 1

        # print(f'STAGE {self.stage_index}, output = {self.output_sample}')


In [138]:
stage0 = radix2_SDF_stage(stage_index=0, size=8)
stage1 = radix2_SDF_stage(stage_index=0, size=4)
stage2 = radix2_SDF_stage(stage_index=0, size=2)


# input_vector = [1.0, 2.0, 3.0, 4.0, 0.0, 0.0, 0.0]
# input_vector = [4.0, 3.0, 2.0, 1.0, 0.0, 0.0, 0.0, 0.0]

# input_vector = [8.0, 2.0, 0.0]
input_vector = [1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
fft_manual = []
for i in range(len(input_vector)):
    stage0.input_sample = input_vector[i]
    stage0.calculate()
    stage1.input_sample = stage0.output_sample
    stage1.calculate()
    stage2.input_sample = stage1.output_sample
    stage2.calculate()
    print(f'i = {i}, real = {stage2.output_sample.real}, imag = {stage2.output_sample.imag}')
    if i > 6:
        fft_manual.append(stage2.output_sample)


i = 0, real = 0.0, imag = 0.0
i = 1, real = 0.0, imag = 0.0
i = 2, real = 0.0, imag = 0.0
i = 3, real = 0.0, imag = 0.0
i = 4, real = 0.0, imag = 0.0
i = 5, real = 0.0, imag = 0.0
i = 6, real = 0.0, imag = 0.0
i = 7, real = 36.0, imag = 0.0
i = 8, real = -4.0, imag = 0.0
i = 9, real = -4.0, imag = 4.0
i = 10, real = -3.9999999999999996, imag = -4.0
i = 11, real = -4.0, imag = 9.65685424949238
i = 12, real = -3.9999999999999996, imag = -1.6568542494923797
i = 13, real = -4.0, imag = 1.6568542494923797
i = 14, real = -3.9999999999999987, imag = -9.65685424949238


In [16]:
def bracewell_buneman(xarray, length, log2length):
    ''' 
    bracewell-buneman bit reversal function
    inputs: xarray is array; length is array length; log2length=log2(length).
    output: bit reversed array xarray. 
    '''
    muplus = int((log2length+1)/2)
    mvar = 1
    reverse = np.zeros(length, dtype = int)
    upper_range = muplus+1
    for _ in np.arange(1, upper_range):
        for kvar in np.arange(0, mvar):
            tvar = 2*reverse[kvar]
            reverse[kvar] = tvar
            reverse[kvar+mvar] = tvar+1
        mvar = mvar+mvar
    if (log2length & 0x01):
            mvar = mvar/2

    mvar = int(mvar)
    for qvar in np.arange(1, mvar):
        
        nprime = qvar-mvar
        rprimeprime = reverse[qvar]*mvar
        for pvar in np.arange(0, reverse[qvar]):
            nprime = nprime+mvar
            rprime = rprimeprime+reverse[pvar]
            temp = xarray[nprime]
            xarray[nprime] = xarray[rprime]
            xarray[rprime] = temp
    return xarray

In [139]:
# print(fft_manual)
fft_manual = np.array(bracewell_buneman(fft_manual, len(fft_manual), int(np.log2(len(fft_manual)))))
print(fft_manual)

[36.+0.j         -4.+9.65685425j -4.+4.j         -4.+1.65685425j
 -4.+0.j         -4.-1.65685425j -4.-4.j         -4.-9.65685425j]


In [113]:
# input_vector = [1.0, 1.0]
# input_vector = [1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0]
input_vector = [1.0, 2.0, 3.0, 4.0]
fft_numpy = np.fft.fft(input_vector)
print(fft_numpy)

[10.+0.j -2.+2.j -2.+0.j -2.-2.j]


In [27]:
print(np.allclose(fft_manual,fft_numpy))

ValueError: operands could not be broadcast together with shapes (3,) (4,) 

## Radix-3 SDF Butterfly 

In [22]:
class radix3_PreAdder:
    """
    A class used to represent a hardware preadder of a radix-3 butterfly

    ...

    Attributes
    ----------
    input_0 : float
        first input of preadder
    input_1 : float
        second input of preadder
    input_2 : float
        third input of preadder
    output_0 : float
        first output
    output_1 : float
        second output
    output_2 : float
        thirs output

    Methods
    -------
    calculate(self)
        Calculates the preadder outputs 
    """
    def __init__(self):
        self.input_0 = 0.0
        self.input_1 = 0.0
        self.input_2 = 0.0
        self.output_0 = 0.0
        self.output_1 = 0.0
        self.output_2 = 0.0


    def calculate(self):
        tmp_0_0 = self.input_0
        tmp_1_0 = self.input_1 + self.input_2
        tmp_2_0 = self.input_1 - self.input_2
        ###### prvi nivo pajplajna ^
        tmp_0_1 = tmp_0_0 + tmp_1_0
        tmp_1_1 = tmp_0_0 - (1/2)*tmp_1_0
        tmp_2_1 = tmp_2_0 * (-1j*np.sqrt(3)/2)
        ###### drugi nivo pajplajna ^
        tmp_0_2 = tmp_0_1
        tmp_1_2 = tmp_1_1 + tmp_2_1
        tmp_2_2 = tmp_1_1 - tmp_2_1
        ###### treci nivo pajplajna ^ (ovo su izlazni registri vrv)
        self.output_0 = tmp_0_2
        self.output_1 = tmp_1_2
        self.output_2 = tmp_2_2

class radix3_Rotator:
    cnt = 0
    def __init__(self, stage_index, num_of_stages, size):
        self.input = 0.0
        self.output = 0.0
        self.stage_index = stage_index
        # self.twiddleROM = (np.ones(3**(num_of_stages-stage_index))).astype(complex)
        self.twiddleROM = (np.ones(size)).astype(complex)
        # self.two_thirds_len = 3**(num_of_stages-stage_index) - (3**(num_of_stages-stage_index)//3)
        self.two_thirds_len = size - size//3
        # print(self.two_thirds_len)
        # N = 3**num_of_stages
        N = size
        if (self.two_thirds_len > 2):
            for i in range(self.two_thirds_len):
                if (i < self.two_thirds_len//2):
                    k = i * 3**(stage_index)
                else:
                    k = 2*(i-self.two_thirds_len//2) * 3**(stage_index)
                # print("k = ", k)
                self.twiddleROM[i+(len(self.twiddleROM) - self.two_thirds_len)] = np.exp(-1j*2*np.pi*k/N)
        
        # print(f"STAGE {stage_index}, twiddle = {self.twiddleROM}")

        # print(self.twiddleROM)

    def rotate(self, fifo_full_flag):
        if (fifo_full_flag): # dozvola da brojac vrti i cita redom twiddle faktore iz memorije
            # print(f"STAGE {self.stage_index}, FIFO FULL")
            self.output = self.input * self.twiddleROM[self.cnt]
            # print(f'STAGE {self.stage_index}, curr twiddle = {self.twiddleROM[self.cnt]}')
            if (self.cnt == len(self.twiddleROM)-1):
                self.cnt = 0
            else:
                self.cnt += 1

In [23]:
class radix3_SDF_stage:
    input_sample = 0.0
    output_sample = 0.0
    op_cnt = 0 # operation counter (counting how many add/subb operations are done)
    def __init__(self, stage_index, num_of_stages, size):
        self.stage_index = stage_index
        self.num_of_stages = num_of_stages
        # self.num_of_samples = 3**(num_of_stages-stage_index)
        self.num_of_samples = size
        # self.fifo_0 = Fifo(3**(num_of_stages-stage_index-1))
        # self.fifo_1 = Fifo(3**(num_of_stages-stage_index-1))
        self.fifo_0 = Fifo(size//3)
        self.fifo_1 = Fifo(size//3)
        self.pre_adder = radix3_PreAdder()
        self.rotator = radix3_Rotator(stage_index=stage_index, num_of_stages=num_of_stages, size=size)

    def isFifoFull_0(self):
        return self.fifo_0.is_full()
    def isFifoFull_1(self):
        return self.fifo_1.is_full()
    
    def calculate(self):
        # print(f'STAGE {self.stage_index}, op_cnt = ', self.op_cnt)
        self.pre_adder.input_0 = self.fifo_0.get_output()
        self.pre_adder.input_1 = self.fifo_1.get_output()
        self.pre_adder.input_2 = self.input_sample
        self.pre_adder.calculate()

        # print(f'STAGE {self.stage_index}, pre_adder_out_0 = {self.pre_adder.output_0}')
        # print(f'STAGE {self.stage_index}, pre_adder_out_1 = {self.pre_adder.output_1}')
        # print(f'STAGE {self.stage_index}, pre_adder_out_2 = {self.pre_adder.output_2}')
        
        if (self.op_cnt < (self.num_of_samples//3)): ## other half of the input stream is comming
            self.output_sample = self.fifo_0.get_output()
            self.fifo_0.shift(self.input_sample)
        elif ((self.op_cnt >= (self.num_of_samples//3)) and (self.op_cnt < (self.num_of_samples*2/3))):
            self.output_sample = self.fifo_1.get_output()
            self.fifo_1.shift(self.input_sample)
        else:
            self.output_sample = self.pre_adder.output_0
            self.fifo_0.shift(self.pre_adder.output_1)
            self.fifo_1.shift(self.pre_adder.output_2)

        self.rotator.input = self.output_sample
        self.rotator.rotate(self.isFifoFull_1())
        self.output_sample = self.rotator.output

        # print(f'stage_{self.stage_index} : {self.output_sample}')

        if (self.op_cnt == self.num_of_samples-1):
            self.op_cnt = 0
        else:
            self.op_cnt += 1

        # print(f'STAGE {self.stage_index}, output = {self.output_sample}')

In [24]:
stage0_radix3 = radix3_SDF_stage(stage_index=0, num_of_stages=3, size=27)
stage1_radix3 = radix3_SDF_stage(stage_index=1, num_of_stages=3, size=9)
stage2_radix3 = radix3_SDF_stage(stage_index=2, num_of_stages=3, size=3)

input_vector = [1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
input_vector = np.random.random(27)
input_vector_padded = np.append(input_vector, np.zeros(26))
fft_radix3_manual = []
for i in range(len(input_vector_padded)):
    stage0_radix3.input_sample = input_vector_padded[i]
    stage0_radix3.calculate()
    stage1_radix3.input_sample = stage0_radix3.output_sample
    stage1_radix3.calculate()
    stage2_radix3.input_sample = stage1_radix3.output_sample
    stage2_radix3.calculate()
    print(stage2_radix3.output_sample)
    if i >= 26:
        fft_radix3_manual.append(stage2_radix3.output_sample)

fft_radix3_manual = np.array(fft_radix3_manual)

0.0
0.0
0j
0j
0j
0j
0j
0j
0j
0j
0j
0j
0j
0j
0j
0j
0j
0j
0j
0j
0j
0j
0j
0j
0j
0j
(15.465465204801635+0j)
(0.47780781134102224-0.7093145723731434j)
(0.47780781134102224+0.7093145723731434j)
(0.21414078068110842+1.8006194466284897j)
(0.597331512931706-0.8585162669248897j)
(-0.5703981812104499+1.2941988967226625j)
(0.2141407806811093-1.80061944662849j)
(-0.5703981812104493-1.2941988967226628j)
(0.5973315129317045+0.8585162669248902j)
(-1.5337253654529122+1.5609074519998867j)
(-0.724018753435885+1.2715721484054645j)
(-1.8928075492131193-1.3867687219543579j)
(-1.085399016912844+0.8655068749693016j)
(-0.27080842249000325-2.044342159148414j)
(0.1336855283935142+2.053740059581184j)
(-0.7564202497592851-0.14666125031594923j)
(0.38349851397855-1.854564584770619j)
(0.8335535142909662-0.8244466818347022j)
(1.2772095741821525+0.9785887863907372j)
(-0.7894299937068492+1.4874521582779519j)
(-0.027147801965072327+0.3596315722525817j)
(1.4748965760940322+0.3688768375671455j)
(-0.5321513365621391+0.46418

In [25]:
import numpy as np

def digit_reverse_array(arr, radix):
    """
    Perform digit-reversal on a NumPy array for a given radix.
    
    Parameters:
        arr (np.ndarray): Input array to be reordered.
        radix (int): The radix (base) for the FFT.
    
    Returns:
        np.ndarray: Reordered array based on digit-reversal indices.
    """
    n = arr.size
    if not np.log(n) / np.log(radix) % 1 == 0:
        raise ValueError("The size of the array must be a power of the radix.")
    
    num_digits = int(np.log(n) / np.log(radix))
    
    def digit_reverse(index, radix, num_digits):
        reversed_index = 0
        for _ in range(num_digits):
            reversed_index = reversed_index * radix + (index % radix)
            index //= radix
        return reversed_index
    
    reordered = np.empty_like(arr)
    for i in range(n):
        reversed_index = digit_reverse(i, radix, num_digits)
        reordered[reversed_index] = arr[i]
    
    return reordered

def reverse_digit_reverse_array(arr, radix):
    """
    Reverse digit-reversal on a NumPy array for a given radix.
    
    Parameters:
        arr (np.ndarray): Input array that was digit-reversed.
        radix (int): The radix (base) used for digit-reversal.
    
    Returns:
        np.ndarray: Array restored to its original order.
    """
    n = arr.size
    if not (np.log(n) / np.log(radix)).is_integer():
        raise ValueError("The size of the array must be a power of the radix.")
    
    num_digits = int(np.log(n) / np.log(radix))
    
    def digit_reverse(index, radix, num_digits):
        reversed_index = 0
        for _ in range(num_digits):
            reversed_index = reversed_index * radix + (index % radix)
            index //= radix
        return reversed_index
    
    reordered = np.empty_like(arr)
    for i in range(n):
        original_index = digit_reverse(i, radix, num_digits)
        reordered[original_index] = arr[i]
    
    return reordered


In [26]:
# print(fft_radix3_manual)
fft_radix3_manual = reverse_digit_reverse_array(fft_radix3_manual, radix=3)
for num in fft_radix3_manual:
    print(num)
# fft_radix3_numpy = np.fft.fft([1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0])
fft_radix3_numpy = np.fft.fft(input_vector)
print("\n")
for num in fft_radix3_numpy:
    print(num)
# print(fft_radix3_numpy)

(15.465465204801635+0j)
(-1.5337253654529122+1.5609074519998867j)
(1.2772095741821525+0.9785887863907372j)
(0.21414078068110842+1.8006194466284897j)
(-1.085399016912844+0.8655068749693016j)
(1.4748965760940322+0.3688768375671455j)
(0.2141407806811093-1.80061944662849j)
(-0.7564202497592851-0.14666125031594923j)
(-1.2476555563475142-1.0371860633933876j)
(0.47780781134102224-0.7093145723731434j)
(-0.724018753435885+1.2715721484054645j)
(-0.7894299937068492+1.4874521582779519j)
(0.597331512931706-0.8585162669248897j)
(-0.27080842249000325-2.044342159148414j)
(-0.5321513365621391+0.46418746755060436j)
(-0.5703981812104493-1.2941988967226628j)
(0.38349851397855-1.854564584770619j)
(-2.6713881524649468-1.3289704107085654j)
(0.47780781134102224+0.7093145723731434j)
(-1.8928075492131193-1.3867687219543579j)
(-0.027147801965072327+0.3596315722525817j)
(-0.5703981812104499+1.2941988967226625j)
(0.1336855283935142+2.053740059581184j)
(-2.1652671505412258-1.7079690805198213j)
(0.5973315129317045+0

In [27]:
twiddles = np.ones(9).astype(complex)
for i in range(6):
    k = i * 3**(0)
    print(k)
    twiddles[i+(9 - 6)] = np.exp(-1j*2*np.pi*k/9)

print(twiddles)


0
1
2
3
4
5
[ 1.        +0.j          1.        +0.j          1.        +0.j
  1.        +0.j          0.76604444-0.64278761j  0.17364818-0.98480775j
 -0.5       -0.8660254j  -0.93969262-0.34202014j -0.93969262+0.34202014j]


## Radix-5 SDF Butterfly 

In [28]:
class radix5_PreAdder:
    def __init__(self):
        self.input_0 = 0.0
        self.input_1 = 0.0
        self.input_2 = 0.0
        self.input_3 = 0.0
        self.input_4 = 0.0
        self.output_0 = 0.0
        self.output_1 = 0.0
        self.output_2 = 0.0
        self.output_3 = 0.0
        self.output_4 = 0.0


    def calculate(self):
        tmp_0_0 = self.input_0
        tmp_1_0 = self.input_1 + self.input_4
        tmp_2_0 = self.input_2 + self.input_3
        tmp_3_0 = self.input_1 - self.input_4
        tmp_4_0 = self.input_2 - self.input_3
        ###### prvi nivo pajplajna ^
        tmp_0_1 = tmp_0_0
        tmp_1_1 = tmp_1_0 + tmp_2_0
        tmp_2_1 = tmp_1_0 - tmp_2_0
        tmp_3_1 = tmp_3_0
        tmp_4_1 = tmp_4_0
        tmp_5_1 = tmp_3_0 + tmp_4_0 # dodatna grana izmedju
        ###### drugi nivo pajplajna ^
        tmp_0_2 = tmp_0_1 + tmp_1_1
        tmp_1_2 = tmp_0_1 + tmp_1_1 * (-0.25)
        tmp_2_2 = tmp_2_1 * 0.559
        tmp_3_2 = tmp_3_1 * (-1j*0.363)
        tmp_4_2 = tmp_4_1 * 1j*1.539
        tmp_5_2 = tmp_5_1 * (-1j*0.588)
        ###### treci nivo pajplajna ^
        tmp_0_3 = tmp_0_2
        tmp_1_3 = tmp_1_2 + tmp_2_2
        tmp_2_3 = tmp_1_2 - tmp_2_2
        tmp_3_3 = tmp_3_2 + tmp_5_2
        tmp_4_3 = tmp_4_2 + tmp_5_2
        ###### cetvrti nivo pajplajna ^
        self.output_0 = tmp_0_3
        self.output_1 = tmp_1_3 + tmp_3_3
        self.output_2 = tmp_2_3 + tmp_4_3
        self.output_4 = tmp_1_3 - tmp_3_3
        self.output_3 = tmp_2_3 - tmp_4_3

class radix5_Rotator:
    cnt = 0
    def __init__(self, stage_index, size):
        self.input = 0.0
        self.output = 0.0
        self.stage_index = stage_index
        # self.twiddleROM = (np.ones(5**(num_of_stages-stage_index))).astype(complex)
        self.twiddleROM = (np.ones(size)).astype(complex)
        # self.four_fifths_len = 5**(num_of_stages-stage_index) - (5**(num_of_stages-stage_index)//5)
        self.four_fifths_len = size - (size//5)
        N = size
        if (self.four_fifths_len > 4):
            for i in range(self.four_fifths_len):
                if (i < self.four_fifths_len/4):
                    k = i * 5**(stage_index)
                elif ((i >= self.four_fifths_len/4) and (i < self.four_fifths_len/2)):
                    # UPITNO
                    k = 2*(i-self.four_fifths_len//4) * 5**(stage_index)
                elif ((i >= self.four_fifths_len/2) and (i < 3*self.four_fifths_len/4)):
                    # UPITNO
                    k = 3*(i-2*self.four_fifths_len//4) * 5**(stage_index)
                else:
                    k = 4*(i-3*self.four_fifths_len//4) * 5**(stage_index)
                # print("k = ", k)
                self.twiddleROM[i+(len(self.twiddleROM) - self.four_fifths_len)] = np.exp(-1j*2*np.pi*k/N)
        
        # print(f"STAGE {stage_index}, twiddle = {self.twiddleROM}")

        # print(self.twiddleROM)

    def rotate(self, fifo_full_flag):
        if (fifo_full_flag): # dozvola da brojac vrti i cita redom twiddle faktore iz memorije
            # print(f"STAGE {self.stage_index}, FIFO FULL")
            self.output = self.input * self.twiddleROM[self.cnt]
            # print(f'STAGE {self.stage_index}, curr twiddle = {self.twiddleROM[self.cnt]}')
            if (self.cnt == len(self.twiddleROM)-1):
                self.cnt = 0
            else:
                self.cnt += 1

In [32]:
class radix5_SDF_stage:
    input_sample = 0.0
    output_sample = 0.0
    op_cnt = 0 # operation counter (counting how many add/subb operations are done)
    def __init__(self, stage_index, size):
        self.stage_index = stage_index
        # self.num_of_stages = num_of_stages
        # self.num_of_samples = 5**(num_of_stages-stage_index)
        self.num_of_samples = size
        # self.fifo_0 = Fifo(5**(num_of_stages-stage_index-1))
        # self.fifo_1 = Fifo(5**(num_of_stages-stage_index-1))
        # self.fifo_2 = Fifo(5**(num_of_stages-stage_index-1))
        # self.fifo_3 = Fifo(5**(num_of_stages-stage_index-1))
        self.fifo_0 = Fifo(size//5)
        self.fifo_1 = Fifo(size//5)
        self.fifo_2 = Fifo(size//5)
        self.fifo_3 = Fifo(size//5)
        self.pre_adder = radix5_PreAdder()
        self.rotator = radix5_Rotator(stage_index=stage_index, size=size)

    def isFifoFull_3(self):
        return self.fifo_3.is_full()
    
    def calculate(self):
        self.pre_adder.input_0 = self.fifo_0.get_output()
        self.pre_adder.input_1 = self.fifo_1.get_output()
        self.pre_adder.input_2 = self.fifo_2.get_output()
        self.pre_adder.input_3 = self.fifo_3.get_output()
        self.pre_adder.input_4 = self.input_sample
        self.pre_adder.calculate()
        
        if (self.op_cnt < (self.num_of_samples//5)): ## other half of the input stream is comming
            self.output_sample = self.fifo_0.get_output()
            self.fifo_0.shift(self.input_sample)
        elif ((self.op_cnt >= (self.num_of_samples//5)) and (self.op_cnt < (self.num_of_samples*2/5))):
            self.output_sample = self.fifo_1.get_output()
            self.fifo_1.shift(self.input_sample)
        elif ((self.op_cnt >= (2*self.num_of_samples//5)) and (self.op_cnt < (self.num_of_samples*3/5))):
            self.output_sample = self.fifo_2.get_output()
            self.fifo_2.shift(self.input_sample)
        elif ((self.op_cnt >= (3*self.num_of_samples//5)) and (self.op_cnt < (self.num_of_samples*4/5))):
            self.output_sample = self.fifo_3.get_output()
            self.fifo_3.shift(self.input_sample)
        else:
            self.output_sample = self.pre_adder.output_0
            self.fifo_0.shift(self.pre_adder.output_1)
            self.fifo_1.shift(self.pre_adder.output_2)
            self.fifo_2.shift(self.pre_adder.output_3)
            self.fifo_3.shift(self.pre_adder.output_4)

        self.rotator.input = self.output_sample
        self.rotator.rotate(self.isFifoFull_3())
        self.output_sample = self.rotator.output

        # print(f'stage_{self.stage_index} : {self.output_sample}')

        if (self.op_cnt == self.num_of_samples-1):
            self.op_cnt = 0
        else:
            self.op_cnt += 1

        # print(f'STAGE {self.stage_index}, output = {self.output_sample}')

In [33]:
stage0_radix5 = radix5_SDF_stage(stage_index=0, size=25)
stage1_radix5 = radix5_SDF_stage(stage_index=1, size=5)

input_vector = [1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
input_vector = np.arange(25)
# input_vector = np.ones(25)
input_vector_padded = np.append(input_vector, np.zeros(24))
fft_radix5_manual = []
for i in range(len(input_vector_padded)):
    stage0_radix5.input_sample = input_vector_padded[i]
    stage0_radix5.calculate()
    stage1_radix5.input_sample = stage0_radix5.output_sample
    stage1_radix5.calculate()
    print(stage1_radix5.output_sample)
    if i >= 24:
        fft_radix5_manual.append(stage1_radix5.output_sample)

fft_radix5_manual = np.array(fft_radix5_manual)

0.0
0.0
0.0
0.0
0j
0j
0j
0j
0j
0j
0j
0j
0j
0j
0j
0j
0j
0j
0j
0j
0j
0j
0j
0j
(300+0j)
(-12.5+17.205j)
(-12.5+4.065j)
(-12.5-4.065j)
(-12.5-17.205j)
(-12.499489407304694+98.94861736848772j)
(-12.500024318943213+13.31164545828936j)
(-12.500092549608448+2.388219716382636j)
(-12.50014423096095-5.885734733940009j)
(-12.500249493182695-22.737747809219712j)
(-12.488686465614304+48.69146547386783j)
(-12.498407812871452+10.345414368991126j)
(-12.500829303804473+0.7937943505603j)
(-12.503042085184134-7.934566608473976j)
(-12.509034332525633-31.571107584945278j)
(-12.509034261577995+31.57082773053025j)
(-12.503041227777663+7.931184551267009j)
(-12.500831726641794-0.7842374196886626j)
(-12.498406447357366-10.350800666952j)
(-12.488686336645182-48.691974195156604j)
(-12.50024949184089+22.7375329929621j)
(-12.500144223968249+5.884615235944018j)
(-12.5000926327116-2.3749152719021893j)
(-12.500024253001657-13.322202410369329j)
(-12.499489398477603-98.9500305466346j)


In [34]:
# print(fft_radix3_manual)

fft_radix5_manual_rev = reverse_digit_reverse_array(fft_radix5_manual, radix=5)

print("Manual fft radix-5")
i = 0
for num in fft_radix5_manual_rev:
    print(f'{i} | {num:.2f}')
    i+=1
# fft_radix3_numpy = np.fft.fft([1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0])
fft_radix5_numpy = np.fft.fft(input_vector)
print("\n")

print("Numpy fft")
i=0
for num in fft_radix5_numpy:
    print(f'{i} | {num:.2f}')
    i+=1
# print(fft_radix3_numpy)

Manual fft radix-5
0 | 300.00+0.00j
1 | -12.50+98.95j
2 | -12.49+48.69j
3 | -12.51+31.57j
4 | -12.50+22.74j
5 | -12.50+17.20j
6 | -12.50+13.31j
7 | -12.50+10.35j
8 | -12.50+7.93j
9 | -12.50+5.88j
10 | -12.50+4.07j
11 | -12.50+2.39j
12 | -12.50+0.79j
13 | -12.50-0.78j
14 | -12.50-2.37j
15 | -12.50-4.07j
16 | -12.50-5.89j
17 | -12.50-7.93j
18 | -12.50-10.35j
19 | -12.50-13.32j
20 | -12.50-17.20j
21 | -12.50-22.74j
22 | -12.51-31.57j
23 | -12.49-48.69j
24 | -12.50-98.95j


Numpy fft
0 | 300.00+0.00j
1 | -12.50+98.95j
2 | -12.50+48.68j
3 | -12.50+31.57j
4 | -12.50+22.74j
5 | -12.50+17.20j
6 | -12.50+13.31j
7 | -12.50+10.34j
8 | -12.50+7.93j
9 | -12.50+5.88j
10 | -12.50+4.06j
11 | -12.50+2.38j
12 | -12.50+0.79j
13 | -12.50-0.79j
14 | -12.50-2.38j
15 | -12.50-4.06j
16 | -12.50-5.88j
17 | -12.50-7.93j
18 | -12.50-10.34j
19 | -12.50-13.31j
20 | -12.50-17.20j
21 | -12.50-22.74j
22 | -12.50-31.57j
23 | -12.50-48.68j
24 | -12.50-98.95j


In [35]:
digit_reversed_data = np.array([0, 5, 10, 15, 20, 1, 6, 11, 16, 21, 2, 7, 12, 17, 22, 3, 8, 13, 18, 23, 4, 9, 14, 19, 24])
radix = 5

# Perform reverse digit-reversal
original_data = reverse_digit_reverse_array(digit_reversed_data, radix)
print("Digit-Reversed Data:", digit_reversed_data)
print("Restored Original Data:", original_data)

Digit-Reversed Data: [ 0  5 10 15 20  1  6 11 16 21  2  7 12 17 22  3  8 13 18 23  4  9 14 19
 24]
Restored Original Data: [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23
 24]


## Digit inversion

In [36]:
def digit_reverse(radices):
    radices_rev = np.copy(radices)
    radices_rev = radices_rev[::-1]

    mr_fft_len = np.prod(radices)
    indices = np.arange(mr_fft_len)

    mult_factors = []
    tmp_len = mr_fft_len
    for radix in radices:
        tmp_len = tmp_len//radix
        mult_factors.append(tmp_len)
    
    mult_factors_rev = []
    tmp_len = mr_fft_len
    for radix in radices_rev:
        tmp_len = tmp_len//radix
        mult_factors_rev.append(tmp_len)
    

    decomp = []
    for index in indices:
        tmp_decomposition = []
        tmp_index = index
        for mult_factor in mult_factors:
            tmp_decomposition.append(tmp_index // mult_factor)
            tmp_index = tmp_index % mult_factor
        decomp.append(tmp_decomposition)
    
    decomp = np.array(decomp)
    decomp = decomp.T[::-1]

    indices_rev = np.multiply(np.tile(mult_factors_rev, (mr_fft_len,1)), decomp.T).sum(axis=1)

    return indices_rev

## Decompositon of N on $2^i \cdot 3^j \cdot 5^k$

The max number of subcarriers in OFDM (5G NR standard) is 3300 $(275 * 12)$, so besides radix 2, 3 and 5, a radix 11 butterfly ($275 = 5^2 \cdot 11^1$) is also needed to achieve the best performance (lowest spectral leakage).
In the next few cells radix powers and the list of all possible FFT sizes will be generated.

It is also possible to avoid the radix 11 butterfly usage if the system is willing to introduce some error because of the spectral leakage. In that case, radices 2, 3 and 5 could generate FFT of size 3840.

In [37]:
i_arr = []
j_arr = []
k_arr = []
for i in range(10):
    for j in range(10):
        for k in range(10):
            if ((((2**i) * (3**j) * (5**k)) <= 275)):
                i_arr.append(i)
                j_arr.append(j)
                k_arr.append(k)

In [39]:
N_arr = []
i_arr = np.unique(i_arr)
j_arr = np.unique(j_arr)
k_arr = np.unique(k_arr)

for i in i_arr:
    for j in j_arr:
        for k in k_arr:
            if((12 * ((2**i) * (3**j) * (5**k))) <= 3300):
            # if (not ((12 * (2**i * 3**j * 5**k)) in N_arr)):
                # print(f"i = {i}, j = {j}, k = {k}")
                N_arr.append(12 * (2**i * 3**j * 5**k))

N_arr = np.array(N_arr)
N_arr.sort()

print(max(N_arr))
print(len(N_arr))

print(N_arr)
print(i_arr)
print(j_arr)
print(k_arr)


3240
53
[  12   24   36   48   60   72   96  108  120  144  180  192  216  240
  288  300  324  360  384  432  480  540  576  600  648  720  768  864
  900  960  972 1080 1152 1200 1296 1440 1500 1536 1620 1728 1800 1920
 1944 2160 2304 2400 2592 2700 2880 2916 3000 3072 3240]
[0 1 2 3 4 5 6 7 8]
[0 1 2 3 4 5]
[0 1 2 3]


# Combining the stages with different radices

## Radix-3 and radix-2

In [41]:
N = 18
test_vector = np.random.random(N)

fft_stage0 = radix3_SDF_stage(stage_index=0, num_of_stages=1, size=18)
fft_stage1 = radix3_SDF_stage(stage_index=0, num_of_stages=1, size=6)
fft_stage2 = radix2_SDF_stage(stage_index=0, num_of_stages=1, size=2)

test_vector_padded = np.append(test_vector, np.zeros(17))
fft_mixed_radix_manual = []
for i in range(len(test_vector_padded)):
    fft_stage0.input_sample = test_vector_padded[i]
    fft_stage0.calculate()
    fft_stage1.input_sample = fft_stage0.output_sample
    fft_stage1.calculate()
    fft_stage2.input_sample = fft_stage1.output_sample
    fft_stage2.calculate()
    print(fft_stage2.output_sample)
    if i >= 17:
        fft_mixed_radix_manual.append(fft_stage2.output_sample)

fft_mixed_radix_manual = np.array(fft_mixed_radix_manual)

radices = [3,3,2]
rev_seq = digit_reverse(radices)

fft_mixed_radix_manual_rev = np.zeros(N).astype(complex)
i = 0
while (i < N):
    fft_mixed_radix_manual_rev[rev_seq[i]] = fft_mixed_radix_manual[i]
    i += 1

0.0
0j
0j
0j
0j
0j
0j
0j
0j
0j
0j
0j
0j
0j
0j
0j
0j
(7.957171425907392+0j)
(2.3171963169231926+0j)
(-1.2292968382714973+0.8141121910976525j)
(0.48791489317407954-0.5809688182958959j)
(0.4879148931740793+0.5809688182958961j)
(-1.229296838271497-0.8141121910976528j)
(0.4076048915718621+0.23633418765420539j)
(1.613032037766342-0.7308239617382677j)
(-0.7743138812851633-2.1614931936724817j)
(0.23436264983481075-0.5341146259013445j)
(-0.5901751214891333-0.44781882327007383j)
(-0.06200582377294589+0.8374850611501791j)
(-0.06200582377294572-0.8374850611501792j)
(-0.5901751214891333+0.44781882327007366j)
(0.23436264983481114+0.5341146259013447j)
(-0.7743138812851641+2.1614931936724817j)
(1.6130320377663423+0.7308239617382681j)
(0.4076048915718624-0.2363341876542055j)


In [42]:
print("Manual fft mixed radix")
i = 0
for num in fft_mixed_radix_manual_rev:
    print(f'{i} | {num:.2f}')
    i+=1

fft_mixed_radix_numpy = np.fft.fft(test_vector)
print("\n")

print("Numpy fft")
i=0
for num in fft_mixed_radix_numpy:
    print(f'{i} | {num:.2f}')
    i+=1

Manual fft mixed radix
0 | 7.96+0.00j
1 | 0.41+0.24j
2 | -0.06-0.84j
3 | -1.23+0.81j
4 | -0.77-2.16j
5 | 0.23+0.53j
6 | 0.49+0.58j
7 | -0.59-0.45j
8 | 1.61+0.73j
9 | 2.32+0.00j
10 | 1.61-0.73j
11 | -0.59+0.45j
12 | 0.49-0.58j
13 | 0.23-0.53j
14 | -0.77+2.16j
15 | -1.23-0.81j
16 | -0.06+0.84j
17 | 0.41-0.24j


Numpy fft
0 | 7.96+0.00j
1 | 0.41+0.24j
2 | -0.06-0.84j
3 | -1.23+0.81j
4 | -0.77-2.16j
5 | 0.23+0.53j
6 | 0.49+0.58j
7 | -0.59-0.45j
8 | 1.61+0.73j
9 | 2.32+0.00j
10 | 1.61-0.73j
11 | -0.59+0.45j
12 | 0.49-0.58j
13 | 0.23-0.53j
14 | -0.77+2.16j
15 | -1.23-0.81j
16 | -0.06+0.84j
17 | 0.41-0.24j


## Radix-5 and radix-2

In [48]:
N = 10
test_vector = np.random.random(N)

radices = [5,2]
rev_seq = digit_reverse(radices)

fft_stage0 = radix5_SDF_stage(stage_index=0, size=10)
fft_stage1 = radix2_SDF_stage(stage_index=0, size=2)

test_vector_padded = np.append(test_vector, np.zeros(9))
fft_mixed_radix_manual = []
for i in range(len(test_vector_padded)):
    fft_stage0.input_sample = test_vector_padded[i]
    fft_stage0.calculate()
    fft_stage1.input_sample = fft_stage0.output_sample
    fft_stage1.calculate()
    print(fft_stage1.output_sample)
    if i >= 9:
        fft_mixed_radix_manual.append(fft_stage1.output_sample)

fft_mixed_radix_manual = np.array(fft_mixed_radix_manual)

fft_mixed_radix_manual_rev = np.zeros(N).astype(complex)
i = 0
while (i < N):
    fft_mixed_radix_manual_rev[rev_seq[i]] = fft_mixed_radix_manual[i]
    i += 1

0.0
0j
0j
0j
0j
0j
0j
0j
0j
(6.354020513197883+0j)
(0.08307300441072574+0j)
(-0.20712788580435473+0.7269126007652883j)
(0.280350394025762+0.833985651977741j)
(1.0701694439064582-1.0367661460135233j)
(-0.09275549134252492+0.41449169052478774j)
(-0.09275549134252492-0.41449169052478796j)
(1.0701694439064582+1.0367661460135236j)
(0.28035039402576206-0.833985651977741j)
(-0.20712788580435476-0.7269126007652883j)


In [49]:
print("Manual fft mixed radix")
# fft_mixed_radix_manual = fft_mixed_radix_manual[rev_seq]
i = 0
i = 0
for num in fft_mixed_radix_manual_rev:
    print(f'{i} | {num:.2f}')
    i+=1

fft_mixed_radix_numpy = np.fft.fft(test_vector)
print("\n")

print("Numpy fft")
i=0
for num in fft_mixed_radix_numpy:
    print(f'{i} | {num:.2f}')
    i+=1

Manual fft mixed radix
0 | 6.35+0.00j
1 | -0.21+0.73j
2 | 1.07-1.04j
3 | -0.09-0.41j
4 | 0.28-0.83j
5 | 0.08+0.00j
6 | 0.28+0.83j
7 | -0.09+0.41j
8 | 1.07+1.04j
9 | -0.21-0.73j


Numpy fft
0 | 6.35+0.00j
1 | -0.21+0.73j
2 | 1.07-1.04j
3 | -0.09-0.41j
4 | 0.28-0.83j
5 | 0.08+0.00j
6 | 0.28+0.83j
7 | -0.09+0.41j
8 | 1.07+1.04j
9 | -0.21-0.73j


# Mixed radix FFT - FXP

In [10]:
import numpy as np
from fxpmath import Fxp
from scipy.fft import fft, ifft

In [11]:
# type of fixed-point numbers used
dtype = 'fxp-s16/8'
DATA = Fxp(None, True, dtype='fxp-s16/8')

## Radix-2 SDF Butterfly FXP

In [12]:
class radix2_PreAdder_fxp:
    """
    A class used to represent a fixed-point hardware preadder of a radix-2 butterfly

    ...

    Attributes
    ----------
    input_a : fxp
        upper input
    input_b : fxp
        lower input
    output_add : fxp
        adder output
    output_sub : fxp
        subtractor output

    Methods
    -------
    calculate(self)
        Calculates the adder and the subtractor outputs 
    """
    def __init__(self):
        self.input_a_r = Fxp(0.0).like(DATA)
        self.input_a_i = Fxp(0.0).like(DATA)

        self.input_b_r = Fxp(0.0).like(DATA)
        self.input_b_i = Fxp(0.0).like(DATA)

        self.output_add_r= Fxp(0.0).like(DATA)
        self.output_add_i= Fxp(0.0).like(DATA)

        self.output_sub_r = Fxp(0.0).like(DATA)
        self.output_sub_i = Fxp(0.0).like(DATA)


    def calculate(self):
        # print("Preadder input a = ", self.input_a_r, self.input_a_i)
        # print("Preadder input b = ", self.input_b_r, self.input_b_i)
        self.output_add_r = self.input_a_r + self.input_b_r
        self.output_add_i = self.input_a_i + self.input_b_i
        self.output_sub_r = self.input_a_r - self.input_b_r
        self.output_sub_i = self.input_a_i - self.input_b_i
        

class radix2_Rotator_fxp:
    cnt = 0
    def __init__(self, stage_index, size):
        self.input_r = Fxp(0.0).like(DATA)
        self.input_i = Fxp(0.0).like(DATA)
        self.output_r = Fxp(0.0).like(DATA)
        self.output_i = Fxp(0.0).like(DATA)
        self.stage_index = stage_index
        # self.twiddleROM = (np.ones(2**(num_of_stages-stage_index))).astype(complex)
        np_rom = np.zeros((size,2))
        np_rom[:,0].fill(1.0)
        
        self.twiddleROM = Fxp(np_rom).like(DATA)
        self.half_len = size//2
        N = size
        for i in range(self.half_len):
            k = i * 2**(stage_index)
            twiddle_factor = np.exp(-1j*2*np.pi*k/N)

            self.twiddleROM[i+self.half_len, 0] = twiddle_factor.real
            self.twiddleROM[i+self.half_len, 1] = twiddle_factor.imag
        
        # print(f"STAGE {stage_index}, twiddle = {self.twiddleROM}")

        # print(self.twiddleROM)

    def rotate(self, fifo_full_flag):
        x_r = Fxp(0.0).like(DATA)
        x_i = Fxp(0.0).like(DATA)
        y_r = Fxp(0.0).like(DATA)
        y_i = Fxp(0.0).like(DATA)
        z_r = Fxp(0.0).like(DATA)
        z_i = Fxp(0.0).like(DATA)
        if (fifo_full_flag): # dozvola da brojac vrti i cita redom twiddle faktore iz memorije
            # print(f"STAGE {self.stage_index}, FIFO FULL")
            x_r(self.input_r)
            x_i(self.input_i)
            y_r(self.twiddleROM[self.cnt, 0])
            y_i(self.twiddleROM[self.cnt, 1])
            # self.output = self.input * self.twiddleROM[self.cnt]
            z_r.set_val(x_r*y_r - x_i*y_i)
            z_i.set_val(x_r*y_i + x_i*y_r)

            self.output_r.set_val(z_r)
            self.output_i.set_val(z_i)
            # print(f'STAGE {self.stage_index}, curr twiddle = {self.twiddleROM[self.cnt]}')
            if (self.cnt == self.half_len*2-1):
                self.cnt = 0
            else:
                self.cnt += 1
        

class Fifo_fxp:
    full = 0
    cnt = 0

    def __init__(self, depth):
        self.depth = depth
        self.buffer = Fxp(np.zeros((depth,2))).like(DATA)
        # print(self.depth)

    def is_full(self):
        return self.full
    
    def get_output(self):
        return self.buffer[-1]
    
    def shift(self, input_sample_r, input_sample_i):
        self.cnt += 1
        # print("FIFO buffer before roll = ", self.buffer)
        self.buffer = np.roll(self.buffer, 1, axis=0).like(DATA)
        # print("FIFO input samples = ", input_sample_r, input_sample_i)
        self.buffer[0,0] = input_sample_r
        self.buffer[0,1] = input_sample_i
        # print("FIFO buffer after roll = ", self.buffer)
        if (self.cnt > self.depth):
            self.full = 1
        else:
            self.full = 0



In [13]:
class radix2_SDF_fxp_stage:
    input_sample_r = Fxp(0.0).like(DATA)
    input_sample_i = Fxp(0.0).like(DATA)
    output_sample_r = Fxp(0.0).like(DATA)
    output_sample_i = Fxp(0.0).like(DATA)
    op_cnt = 0 # operation counter (counting how many add/subb operations are done)
    def __init__(self, stage_index, size):
        self.stage_index = stage_index
        self.size = size
        # self.num_of_stages = num_of_stages
        # self.num_of_samples = 2**(num_of_stages-stage_index)
        self.num_of_samples = size
        self.fifo = Fifo_fxp(size//2)
        self.pre_adder = radix2_PreAdder_fxp()
        self.rotator = radix2_Rotator_fxp(stage_index=stage_index, size=size)

    def isFifoFull(self):
        return self.fifo.is_full()
    
    def calculate(self):
        # print(f'STAGE {self.stage_index}, op_cnt = ', self.op_cnt)
        fifo_out_reg = self.fifo.get_output()
        # print("AAA = ", fifo_out_reg)
        self.pre_adder.input_a_r.set_val(fifo_out_reg[0])
        self.pre_adder.input_a_i.set_val(fifo_out_reg[1])
        self.pre_adder.input_b_r.set_val(self.input_sample_r)
        self.pre_adder.input_b_i.set_val(self.input_sample_i)
        self.pre_adder.calculate()

        # print(f'STAGE {self.size}, add_out r = {self.pre_adder.output_add_r} i = {self.pre_adder.output_add_i}')
        # print(f'STAGE {self.size}, sub_out r = {self.pre_adder.output_sub_r} i = {self.pre_adder.output_sub_i}')
        # print(f'STAGE {self.size}, sub_out = {self.pre_adder.output_sub}')
        
        if (self.op_cnt//(self.num_of_samples/2)): ## other half of the input stream is comming
            self.output_sample_r.set_val(self.pre_adder.output_add_r)
            self.output_sample_i.set_val(self.pre_adder.output_add_i)
            self.fifo.shift(self.pre_adder.output_sub_r, self.pre_adder.output_sub_i) 
        else:
            fifo_out_reg = self.fifo.get_output()
            self.output_sample_r.set_val(fifo_out_reg[0])
            self.output_sample_i.set_val(fifo_out_reg[1])
            self.fifo.shift(self.input_sample_r, self.input_sample_i)

        self.rotator.input_r.set_val(self.output_sample_r)
        self.rotator.input_i.set_val(self.output_sample_i)
        self.rotator.rotate(self.isFifoFull())
        self.output_sample_r.set_val(self.rotator.output_r)
        self.output_sample_i.set_val(self.rotator.output_i)

        # print(f'stage_{self.stage_index} : {self.output_sample}')

        if (self.op_cnt == self.num_of_samples-1):
            self.op_cnt = 0
        else:
            self.op_cnt += 1

        # print(f'STAGE {self.stage_index}, output = {self.output_sample}')


In [14]:
stage0 = radix2_SDF_fxp_stage(stage_index=0, size=8)
stage1 = radix2_SDF_fxp_stage(stage_index=0, size=4)
stage2 = radix2_SDF_fxp_stage(stage_index=0, size=2)


# input_vector = [1.0, 2.0, 3.0, 4.0, 0.0, 0.0, 0.0]
# input_vector = [4.0, 3.0, 2.0, 1.0, 0.0, 0.0, 0.0, 0.0]

# input_vector = [1.0, 1.0, 0.0]
input_vector_r = Fxp([1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]).like(DATA)
input_vector_i = Fxp(np.zeros(15)).like(DATA)
fft_manual_fxp = []
for i in range(len(input_vector_r)):
    stage0.input_sample_r.set_val(input_vector_r[i])
    stage0.input_sample_i.set_val(input_vector_i[i])
    stage0.calculate()
    stage1.input_sample_r.set_val(stage0.output_sample_r)
    stage1.input_sample_i.set_val(stage0.output_sample_i)
    stage1.calculate()
    stage2.input_sample_r.set_val(stage1.output_sample_r)
    stage2.input_sample_i.set_val(stage1.output_sample_i)
    stage2.calculate()
    print(f'i = {i}, real = {stage2.output_sample_r}, imag = {stage2.output_sample_i}')
    if i > 6:
        fft_manual_fxp.append(complex(stage2.output_sample_r, stage2.output_sample_i))

fft_manual_fxp = np.array(fft_manual_fxp)

i = 0, real = 0.0, imag = 0.0
i = 1, real = 0.0, imag = 0.0
i = 2, real = 0.0, imag = 0.0
i = 3, real = 0.0, imag = 0.0
i = 4, real = 0.0, imag = 0.0
i = 5, real = 0.0, imag = 0.0
i = 6, real = 0.0, imag = 0.0
i = 7, real = 36.0, imag = 0.0
i = 8, real = -4.0, imag = 0.0
i = 9, real = -4.0, imag = 4.0
i = 10, real = -4.0, imag = -4.0
i = 11, real = -4.0, imag = 9.65625
i = 12, real = -4.0, imag = -1.65625
i = 13, real = -4.0, imag = 1.65625
i = 14, real = -4.0, imag = -9.65625


In [17]:
fft_manual_fxp = np.array(bracewell_buneman(fft_manual_fxp, len(fft_manual_fxp), int(np.log2(len(fft_manual_fxp)))))
print(fft_manual_fxp)

[36.+0.j      -4.+9.65625j -4.+4.j      -4.+1.65625j -4.+0.j
 -4.-1.65625j -4.-4.j      -4.-9.65625j]


In [21]:
input_vector_float = [1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0]
fft_numpy = np.fft.fft(input_vector_float)
print(fft_numpy)

[36.+0.j         -4.+9.65685425j -4.+4.j         -4.+1.65685425j
 -4.+0.j         -4.-1.65685425j -4.-4.j         -4.-9.65685425j]


## Radix-3 SDF Butterfly FXP

## SQNR evaluation

In [22]:
import numpy as np
from scipy.signal import chirp
from scipy.fftpack import fft

# Fixed-point conversion parameters
N_WORD = 16  # Total number of bits
N_FRAC = 12  # Fractional bits (Q-format: Q(N_WORD-N_FRAC).N_FRAC)

# FFT size
N = 8  

def float_to_fixed(x, N_WORD, N_FRAC):
    """Convert floating-point values to fixed-point representation."""
    scale = 2**N_FRAC
    x_fixed = np.round(x * scale).astype(np.int16)  # Convert to int16
    return x_fixed

def fixed_to_float(x_fixed, N_FRAC):
    """Convert fixed-point values back to floating-point."""
    return x_fixed / (2**N_FRAC)

def generate_sine_wave(N, k, A=1.0):
    """ Generate a sinusoidal test signal """
    n = np.arange(N)
    x = A * np.sin(2 * np.pi * k * n / N)
    return x

def generate_multitone(N, freqs, A=1.0):
    """ Generate a multi-tone test signal """
    n = np.arange(N)
    x = sum(A * np.sin(2 * np.pi * f * n / N) for f in freqs)
    x /= max(abs(x))  # Normalize to avoid overflow
    return x

def generate_white_noise(N, A=1.0):
    """ Generate a white noise signal """
    x = A * (2 * np.random.rand(N) - 1)  # Uniform noise in range [-A, A]
    return x

def generate_chirp(N, f0, f1, A=1.0):
    """ Generate a linear chirp signal """
    t = np.linspace(0, 1, N)
    x = A * chirp(t, f0=f0, f1=f1, t1=1, method='linear')
    return x

def compute_sqnr(x_float, x_fixed):
    """ Compute SQNR between floating-point and fixed-point FFT results """
    P_signal = np.mean(np.abs(x_float) ** 2)
    P_noise = np.mean(np.abs(x_float - x_fixed) ** 2)
    
    SQNR = 10 * np.log10(P_signal / P_noise) if P_noise > 0 else np.inf
    return SQNR

# Select test signal
test_signal = 'sine'  # Options: 'sine', 'multi', 'noise', 'chirp'

if test_signal == 'sine':
    x = generate_sine_wave(N, k=5, A=0.9)
elif test_signal == 'multi':
    x = generate_multitone(N, [3, 7, 15], A=0.9)
elif test_signal == 'noise':
    x = generate_white_noise(N, A=0.9)
elif test_signal == 'chirp':
    x = generate_chirp(N, f0=1, f1=30, A=0.9)

# Convert to fixed-point
x_fixed = float_to_fixed(x, N_WORD, N_FRAC)

# Compute FFT (floating-point)
X_float = fft(x)

# Compute FFT (fixed-point)
X_fixed = fft(fixed_to_float(x_fixed, N_FRAC))

# Compute SQNR
sqnr_value = compute_sqnr(X_float, X_fixed)
print(f"SQNR for {test_signal} signal: {sqnr_value:.2f} dB")


SQNR for sine signal: 78.70 dB


In [23]:
# Compute SQNR
sqnr_value = compute_sqnr(fft_numpy, fft_manual_fxp)
print(f"SQNR for signal: {sqnr_value:.2f} dB")

SQNR for signal: 90.48 dB
